In [89]:
!pip install gensim

In [90]:
import pandas as pd
import numpy as np
# from func import clean_df
import re, string, keras

# model imports
from keras.layers import Dense, Dropout
from keras.models import Sequential
import keras.layers as layers
import tensorflow as tf

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# NLP imports
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from gensim.models import Word2Vec
from sklearn.preprocessing import OneHotEncoder

In [92]:
data = pd.read_csv('./dataset/data.csv', quotechar='"', escapechar='\\') # read the dataset
print(len(data))
print(data['subject'].unique())
data.sample(10)

39942
['politicsNews' 'worldnews' 'News' 'politics' 'Government News'
 'left-news']


,label,title,text,subject,date
8303,1,Obama highlights climate progress at home befo...,HONOLULU (Reuters) - Preserving natural places...,politicsNews,"August 31, 2016"
20215,0,Man In ‘I Stand For The National Anthem’ Shir...,You know how Trump supporters are screaming be...,News,"October 16, 2017"
20454,0,Trump’s Involvement In Houston Chemical Plant...,In the aftermath of the historic flooding that...,News,"September 1, 2017"
215,1,Nuclear plan backer denies Inauguration Day te...,WASHINGTON (Reuters) - A company promoting a p...,politicsNews,"December 11, 2017"
21610,0,‘Grow The F*ck Up!’: Trump Gets Taken To The ...,"Just yesterday, amateur president Donald Trump...",News,"May 2, 2017"
34738,0,THE 2016 Calendar That Won’t Be Hanging In ISI...,Vladimir Putin the anti-Obama Fans of the mave...,politics,"Dec 28, 2015"
31407,0,HILARIOUS! Look Who LIBERAL Middlebury Profess...,"Two weeks ago at Middlebury College, Charles M...",politics,"Mar 14, 2017"
1815,1,Senate panel rejects Trump's 'doctrine of retr...,WASHINGTON (Reuters) - A powerful Senate commi...,politicsNews,"September 8, 2017"
1451,1,Factbox: Trump on Twitter (Oct 1) - Rex Tiller...,The following statements were posted to the ve...,politicsNews,"October 2, 2017"
33845,0,HERE’S WHY TRUMP’S Muslim Ban Is Absolutely Ne...,The biggest thing a president can do for the p...,politics,"May 23, 2016"


In [93]:
data = clean_df(df=data, drop_columns=['text','date','reporter','subject']) # clean the dataset
print(len(data))
data.sample(10)

Rows dropped: 703
39239


,label,title,content,year,month,day
22054,0,Trump’s Rambling Speech About Abe Lincoln Is ...,I m convinced that whenever Trump says most ...,2017,3,22
33693,0,TAKE A LOOK INSIDE THE NEW “PUTIN CAFE” Featur...,The majority of Americans would likely find th...,2016,4,13
3237,1,U.S. strikes against pro-Syrian government for...,U.S. Defense Secretary Jim Mattis said on Tues...,2017,6,13
36965,0,NFL NIGHTMARE CONTINUES…Seahawks Player Caught...,In case anyone was confused about how the whol...,2017,12,16
11024,1,The frequent-flyer U.S. Congress: lawmakers wo...,Anyone seeking a table at Carmine’s Italian re...,2016,2,1
33377,0,ANTI-AMERICAN DEM REP: The Declaration Of Inde...,You will NOT believe this woman! Barbara Norto...,2016,5,28
3389,1,U.S. embassy in Egypt bans personnel from visi...,Personnel assigned to the U.S. Mission in Egyp...,2017,6,6
15679,1,"Car bomb explosion kills, injures dozens in Sy...",A suicide car bomb exploded on Saturday in Syr...,2017,11,4
17193,1,Xi says China will oppose any behaviors that t...,China will safeguard its sovereignty and secur...,2017,10,18
31017,0,CAUGHT ON VIDEO: Tour Bus Passenger Wrestles L...,A brave London tour bus passenger saw a Muslim...,2017,4,29


In [94]:
x_data_lst = ['title','subject','reporter','content']

In [95]:
import nltk
# download the stopwords
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [96]:
# encoding the labels

def tokeniseEmbed_and_oneHot(df: pd.DataFrame, tokenised_columns: list[str], oneHotEnc_columns: list[str]):

    def remove_punctuation(tokenised_text:str) -> list[str]:
        '''
        Remove punctuation from the tokenized text.
        Args:
            tokenised_text (str): The tokenized text from which punctuation needs to be removed.
        '''
        pattern = re.compile('[%s]' % re.escape(string.punctuation))
        new_sentence = []
        for token in tokenised_text:
            new_token = pattern.sub(u'', token) # Replace by an empty string
            if not new_token == u'':
                new_sentence.append(new_token) # Append only tokens which are not empty
        return new_sentence

    def tokenise_columns(df: pd.DataFrame, tokenised_columns: list[str]):
        stop_words = set(stopwords.words('english'))
        lemmatizer = WordNetLemmatizer()
        for column in tokenised_columns:
            temp_col = []
            for text in df[column].values:
                filtered_tokens = []
                # Tokenize & remove punctuation from the text
                tokenised_text = remove_punctuation(word_tokenize(text))
                # Remove stop words
                for token in tokenised_text:
                    if token.lower() not in stop_words:
                        filtered_tokens.append(token.lower())
                # Apply lemmatization
                lemmatized_tokens = [lemmatizer.lemmatize(token) for token in filtered_tokens]
                temp_col.append(lemmatized_tokens)
            df[column] = pd.Series(temp_col)
        return df

    def embed_tokenised_text(df: pd.DataFrame, tokenised_columns: list[str]) -> pd.DataFrame:
      for column in tokenised_columns:
          model = Word2Vec(sentences=df[column].values, vector_size=100, window=5, min_count=1, workers=4)
          df[column] = df[column].apply(
              lambda tokens: np.mean([model.wv[token] for token in tokens if token in model.wv], axis=0)
              if tokens else np.zeros(100)
          )
      return df

    def oneHotEncode_columns(df: pd.DataFrame, oneHotEnc_columns: list[str]) -> pd.DataFrame:
        encoder = OneHotEncoder(sparse_output=False)
        for column in oneHotEnc_columns:
            encoded_data = encoder.fit_transform(df[[column]])
            encoded_df = pd.DataFrame(encoded_data, columns=[f"{column}_{cat}" for cat in encoder.categories_[0]])
            df = pd.concat([df, encoded_df], axis=1).drop(column, axis=1)
        return df

    df = tokenise_columns(df, tokenised_columns)
    df = embed_tokenised_text(df, tokenised_columns)
    df = oneHotEncode_columns(df, oneHotEnc_columns)
    return df


tkn_cols = ['content','title']
oneHot_cols = []

data = tokeniseEmbed_and_oneHot(data, tkn_cols, oneHot_cols)
data.to_csv('data-processed.csv', index=False)
# try:
#   pd.read_csv('data-processed.csv')
# except:
#   data = tokeniseEmbed_and_oneHot(data, tkn_cols, oneHot_cols)
#   data.to_csv('data-processed.csv', index=False)

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39239 entries, 0 to 39238
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    39239 non-null  int64 
 1   title    39239 non-null  object
 2   content  39239 non-null  object
 3   year     39239 non-null  Int64 
 4   month    39239 non-null  Int64 
 5   day      39239 non-null  Int64 
dtypes: Int64(3), int64(1), object(2)
memory usage: 1.9+ MB


In [97]:
data.sample(6)

,label,title,content,year,month,day
13177,1,"[0.078750625, 0.03534115, -0.11033996, -0.0358...","[-1.5241735, -0.24578986, -0.7554223, -0.36968...",2017,12,4
27554,0,"[-0.18597697, -0.15005696, 0.016338544, 0.0194...","[-0.9925788, 0.67208713, -0.23085901, 0.107601...",2016,3,11
38313,0,"[-0.16496359, 0.01066801, -0.052444007, 0.0512...","[-0.6235265, 1.0292363, 0.34358314, -0.3295572...",2017,4,4
13031,1,"[-0.15151231, 0.02925433, 0.23294495, -0.16524...","[-0.2991243, 0.85253197, -0.48740098, -0.11598...",2017,12,6
4238,1,"[0.09423668, 0.04198975, -0.019752026, 0.04635...","[-0.68685037, 0.44129598, -0.31771195, 0.01323...",2017,4,14
27846,0,"[-0.22786964, -0.061803475, -0.011827225, 0.03...","[-0.784594, 0.513224, -0.4295842, 0.17638291, ...",2016,2,26


In [98]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

def prepare_features(df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    '''
    Separate features and target, scale numeric columns,
    and stack embedded columns into a single feature matrix.
    Args:
        df (pd.DataFrame): The fully preprocessed dataframe.
    Returns:
        X (np.ndarray): Feature matrix.
        y (np.ndarray): Target labels.
    '''
    df = df.copy()
    y = df['label'].values
    df = df.drop(columns=['label'])

    # Columns that contain Word2Vec embedding arrays need to be stacked
    # All other columns are treated as scalar features (OHE, year, month, day)
    embedding_cols = ['content', 'title']
    scalar_cols = [col for col in df.columns if col not in embedding_cols]

    # Stack embedding arrays into 2D matrices then concatenate
    embedded = np.hstack([
        np.vstack(df[col].values) for col in embedding_cols if col in df.columns
    ])

    # Scale scalar features
    scaler = StandardScaler()
    scaled_scalars = scaler.fit_transform(df[scalar_cols].values.astype(float))

    X = np.hstack([embedded, scaled_scalars])
    return X, y


def build_model(input_dim: int) -> keras.Model:
    '''
    Build a baseline dense neural network for binary classification.
    Args:
        input_dim (int): Number of input features.
    Returns:
        model (keras.Model): Compiled Keras model.
    '''
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),

        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),

        layers.Dense(1, activation='sigmoid')  # binary output
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    return model


def train_model(df: pd.DataFrame) -> keras.Model:
    '''
    Full pipeline: prepare features, split data, build and train model.
    Args:
        df (pd.DataFrame): The fully preprocessed dataframe.
    Returns:
        model (keras.Model): Trained Keras model.
    '''
    X, y = prepare_features(df)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    model = build_model(input_dim=X_train.shape[1])
    model.summary()

    callbacks = [
        # Stop early if val_loss stops improving
        keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=5, restore_best_weights=True
        ),
        # Reduce LR if val_loss plateaus
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
        )
    ]

    history = model.fit(
        X_train, y_train,
        validation_split=0.1,
        epochs=50,
        batch_size=64,
        callbacks=callbacks,
        verbose=1
    )

    # Final evaluation on held-out test set
    loss, accuracy, auc = model.evaluate(X_test, y_test, verbose=0)
    print(f'\nTest Accuracy: {accuracy:.4f}  |  AUC: {auc:.4f}  |  Loss: {loss:.4f}')

    return model, history


# --- Run ---
# Assumes `data` is your fully preprocessed dataframe from tokeniseEmbed_and_oneHot
model, history = train_model(data)

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_16 (Dense)                │ (None, 256)            │        52,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 94,977 (371.00 KB)

 Trainable params: 94,209 (368.00 KB)

 Non-trainable params: 768 (3.00 KB)

Epoch 1/50
442/442 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.9543 - auc: 0.9911 - loss: 0.1175 - val_accuracy: 0.9611 - val_auc: 0.9982 - val_loss: 0.0932 - learning_rate: 0.0010
Epoch 2/50
442/442 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9774 - auc: 0.9969 - loss: 0.0624 - val_accuracy: 0.9857 - val_auc: 0.9990 - val_loss: 0.0428 - learning_rate: 0.0010
Epoch 3/50
442/442 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9822 - auc: 0.9978 - loss: 0.0518 - val_accuracy: 0.9879 - val_auc: 0.9991 - val_loss: 0.0325 - learning_rate: 0.0010
Epoch 4/50
442/442 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9848 - auc: 0.9985 - loss: 0.0416 - val_accuracy: 0.9901 - val_auc: 0.9992 - val_loss: 0.0294 - learning_rate: 0.0010
Epoch 5/50
442/442 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9860 - auc: 0.9986 - loss: 0.0408 - val_accuracy: 0.9904 - val_auc: 0.9992 - val_loss: 0.0289 - learning_rate: 0.0010
Epoch 6/50
442/442 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9862 - auc: 